# A3 移植 (2/8)：M1 —— int8 C8 直读桥

> **一句话结论**：A3 做不了 fp8 块量化，NoPE 改用 int8；注意力侧按 dtype 与算子可得性分流，fp8 走定制 `_vllm` 算子，int8 回落 vendor 原生量化稀疏注意力。

## 1. 问题是什么

A5 的 C8 行 NoPE 是原生 E4M3FN fp8。A3 上请求 `dst_type=fp8` 直接报 161002（不支持 fp8 block quant），所以 NoPE 分量改成 **int8**，scales 仍用 FP32。

## 2. 改了哪三处

1. **dtype 门放宽**：C8 cache dtype 从只接受 `E4M3FN` 放宽到 `(E4M3FN, int8)`；
2. **注意力 dispatch**：
   - 若 `_vllm` 定制算子存在且 dtype 是 fp8，走 `npu_kv_quant_sparse_flash_attention_vllm`；
   - 否则回落 vendor `torch_npu.npu_kv_quant_sparse_flash_attention`；
3. **c8 视图/打包**：`pack_c8_components` / `c8_storage_view` 同步接受 int8。

## 3. int8 回落路径的契约

vendor 原生算子按 **TND 布局 + 绝对物理行号 + `key_quant_mode=2`** 工作；行号契约与上游一致，`-1` 仍表示空行掩码。

## 4. 探针验证结论

单卡探针确认：int8 656B 行 + 空洞 slab + `-1` 掩码的 SFA 路径 PASS，max_err 0.012、cos 1.0，图捕获 replay 误差 0.0000。

## 5. 收益（4k bs16）

- 步时：170.8 → 128.1 ms/步；
- TTFT：17.3 → 9.8 s。

## 6. 小结

- M1 = dtype 门放宽 + 注意力分流 + 视图适配；
- int8 走 vendor 原生算子，行号/掩码契约不变；
- 下一篇讲 M2：resolver 从 SIMT 搬到 AICPU。

**下一篇**：`03-m2-aicpu-resolver.ipynb`
